# Introduction

This project aims to improve the efficiency of robust simulation-based inference, proposed by Huang et al. in 2023. Specifically, it does so by improving on the maximum-mean-discrepancy (MMD) metric that the previous paper uses to regularize the model summarizers. We investigate the implementation of sample-efficient MMD and quasi-Monte Carlo (QMC) methods to improve the computation of this metric. 

In [1]:
import sys
sys.path.append('../')
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pickle
import matplotlib
import matplotlib.pyplot as plt
# import seaborn as sns
from itertools import permutations
import random
import time
import os

from utils.metrics import RMSE
import utils.metrics as metrics
from simulators.ricker import ricker

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

import warnings
warnings.filterwarnings('ignore')

# Below: new imports
from utils.timer import Timer

print(device)

c:\Users\azhao\.conda\envs\efficient-sbi\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\azhao\.conda\envs\efficient-sbi\lib\site-packages\arviz\data\base.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


cuda:0


# Precursor: code and model definitions

In [106]:
# General parameters

batch_size = 256

### Define approximate Bayesian computation (ABC) model

From Huang et al. (2023)

In [107]:
def regression_ABC(s_obs, param, sumStats, p):
    def mad(data):
        return np.mean(np.abs(data - np.mean(data, axis=0)), axis=0)

    if param.shape[0] < param.shape[1]:
        param = np.transpose(param)

    if sumStats.shape[0] < sumStats.shape[1]:
        sumStats = np.transpose(sumStats)
    
    M = len(param)
    M_epsilon = int(M*p)
    sumStats = sumStats
    s_obs = s_obs

    norm_factor = mad(sumStats)

    norm_sumStats = sumStats / norm_factor
    norm_s_obs = s_obs / norm_factor

    distance = np.linalg.norm(norm_sumStats - norm_s_obs, axis = 1)
    max_accepted_distance = np.sort(distance)[M_epsilon - 1]

    posterior_samples = param[distance <= max_accepted_distance, :]
    norm_sumStats_star = norm_sumStats[distance <= max_accepted_distance, :]

    weights = 1 - (distance[distance <= max_accepted_distance] / max_accepted_distance)**2
    W = np.diag(weights)

    s_obs_norm = np.tile(norm_s_obs, (M_epsilon,1))
    X = np.column_stack((np.ones(shape = (M_epsilon,1)), norm_sumStats_star - s_obs_norm))

    A = np.matmul(X.T, W)

    solution = np.linalg.solve(np.matmul(A, X), np.matmul(A, posterior_samples))

    beta = solution[1:,:]

    posterior_samples_adjusted = posterior_samples - np.matmul((norm_sumStats_star - s_obs_norm), beta)

    return posterior_samples_adjusted
    

### Define Ricker summary network

In [108]:
class RickerSummary(nn.Module):
    def __init__(self, input_size, hidden_dim, num_realizations=100):
        super(RickerSummary, self).__init__()

        self.hidden_dim = hidden_dim
        self.input_size = input_size
        self.N = num_realizations
        
        self.encoder = nn.Sequential(nn.Conv1d(self.input_size, 4, 3, 4),
                                     nn.Conv1d(4, 4, 3, 4),
                                     nn.Conv1d(4, 4, 3, 4),
                                     )
        
        self.decoder = nn.Sequential(nn.ConvTranspose1d(4, 4, 3, 4),
                                     nn.ConvTranspose1d(4, 4, 3, 4),
                                     nn.ConvTranspose1d(4, self.input_size, 3, 4),
                                     nn.Upsample(100)
                                     )

    def forward(self, Y):
        embeddings = self.encoder(Y.reshape(-1, 1, 100))
        output = self.decoder(embeddings.reshape(-1, 4, 1)).reshape(-1, self.N, 100)
        return output
    
    def forward_encoder(self, Y):
        embeddings = self.encoder(Y.reshape(-1, 1, 100)).reshape(-1, self.N, 4)
        return embeddings

### Define solver for ABC

In order to evaluate each method, we need to solve the best summary statistics and produce a posterior distribution. We can define a modular solver that uses different methods to produce our output. We can use the aforementioned `RickerSummary()` to define the solver.

In [109]:
def solve_abc(x, beta, obs_cont, m, N, t=100, u=None, mmd_metric=None, root=".", sample_size=256, max_epochs=30, patience=3, lr=0.01, verbose=False):
    """
    General solver function for Ricker summary statistics. 
    Use mmd_metric to differentiate between robust and efficient-robust implementations.
    Set beta to 0 for a non-regularized "normal" solve.
    Timer laps once per epoch.
    Dataloader is automatically selected by x

    Args:
        x (tensor): input data x (x or x_qmc)
        beta (float): regularization coefficient
        obs_cont (tensor): tensor containing contaminated observations, generated with generate_data
        m (int): number of simulations
        N (int): number of realizations
        u (tensor): u matrix
        t (int): simulation time
        mmd_metric (Metric): either None, MMD_unweighted, or MMD_weighted
        root (string): output directory
        sample_size (int): number of samples 
        max_epochs (int): maximum number of epochs to run
        patience (int): number of epochs without descent to terminate training sooner
        lr (float): learning rate
        verbose (bool): whether or not to print results
    """
    summary_net = RickerSummary(1, 4, num_realizations=N).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(summary_net.parameters(), lr=lr)

    index_list = [int(i) for i in range(len(x))]

    name = f"{mmd_metric.__name__ if mmd_metric is not None else 'normal'}_lambda={beta}"

    dataloader = DataLoader(x, batch_size=batch_size, shuffle=True)

    timer = Timer(name, root)
    timer.start()
    previous_loss = np.inf
    epochs_without_descent = 0
    for epoch in range(max_epochs):
        running_loss = 0.0

        for data in dataloader:
            X = data
            optimizer.zero_grad()

            Y = summary_net(X)

            random.shuffle(index_list)

            xx = x[index_list[:sample_size]]

            uu = u[index_list[:sample_size]] if u is not None else None


            context_embeddings = torch.mean(summary_net.forward_encoder(xx), dim=1)
            obs_embeddings = torch.mean(summary_net.forward_encoder(obs_cont), dim=1)

            l_scale = metrics.median_heuristic(context_embeddings)


            ae_loss = criterion(Y, X) / (m * t)

            if (beta > 0):
                if (mmd_metric is None):
                    raise TypeError("If beta is greater than 0, a regularizing metric must be specified: should be one of MMD_weighted, MMD_unweighted")
                
                metric_args = [
                    context_embeddings,
                    obs_embeddings
                ]

                if (uu is not None):
                    u_flat = uu.reshape(uu.shape[0], -1).float()
                    z = metrics.embedding_Gaussian(u_flat)
                    w = metrics.computeWeights(u_flat, z)
                    metric_args.append(w)
                
                metric_args.append(l_scale)

                summary_loss = mmd_metric(*metric_args)

                loss = ae_loss + beta*summary_loss
            else:
                loss = ae_loss

            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        timer.lap(verbose=verbose)

        # use a patience of 0 to remove early stopping
        if (patience > 0):
            if (running_loss > previous_loss):
                epochs_without_descent += 1
            else:
                previous_loss = running_loss
                epochs_without_descent = 0
            
            if (epochs_without_descent > patience):
                break
        if (verbose):
            print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader)}")
    timer.stop()
    return summary_net

### Define solver for NPE

Replicating `exp_ricker.py`

In [110]:
# from networks.summary_nets import RickerSummary
# from utils.get_nn_models import *
# from inference.snpe.snpe_c import SNPE_C
# from inference.base import *
# from utils.torchutils import *
# from scipy import stats as stats
# from utils.user_input_checks import process_prior
# from utils.metrics import *
# import pickle

# def solve_ricker_npe(*, beta=1.0, degree=0.2, distance="mmd", m=1024, n=128, theta_true=[4, 10], prior_mismatch=False, root="../objects/NPE/ricker/", sample_size=256, qmc=False):
#     task_name = f"degree={degree}_{distance}_beta={beta}_theta={theta_true}_num={m}_size={sample_size}"
#     root_name = root + str(task_name)
#     timer = Timer(task_name, root_name)

#     if not os.path.exists(root_name):
#         os.makedirs(root_name)

#     if prior_mismatch:
#         prior = [Uniform(2 * torch.ones(1, device=device), 
#                          8 * torch.ones(1, device=device)),
#                  torch.distributions.log_normal.LogNormal(
#                         loc=torch.tensor([0.5], device=device), 
#                         scale=torch.tensor([1], device=device)
#                     )]
#     else:
#         prior = [Uniform(2 * torch.ones(1, device=device), 
#                          8 * torch.ones(1, device=device)),
#                  Uniform(torch.zeros(1, device=device), 
#                          20 * torch.ones(1, device=device))]

#     prior, _, _ = process_prior(prior)

#     sum_net = RickerSummary(input_size=1, hidden_dim=4).to(device)
#     neural_posterior = posterior_nn(
#         model="maf",
#         embedding_net=sum_net,
#         hidden_features=20,
#         num_transforms=3
#     )

#     inference = SNPE_C(prior=prior, density_estimator=neural_posterior, device=str(device))

#     if prior_mismatch:
#         obs_cont = torch.tensor(np.load("data/ricker_obs_pm.npy")).reshape(-1, n, 100).to(device)
#     else:
#         obs_cont = torch.tensor(np.load(f"data/ricker_obs_{int(degree * 10)}.npy")).to(device)

#     suffix = "_qmc" if qmc else ""
        
#     if prior_mismatch:
#         theta = torch.tensor(np.load(f"data/ricker_theta_{m}_pm.npy")).to(device)
#         x = torch.tensor(np.load(f"data/ricker_x_{m}_pm.npy")).reshape(m, n, 100).to(device)
#         u = torch.tensor(np.load(f"data/ricker_u_{m}_pm.npy")).reshape(m, n, 100).to(device)
#     else:
#         theta = torch.tensor(np.load(f"data/ricker_theta_{m}{suffix}.npy")).to(device)
#         x = torch.tensor(np.load(f"data/ricker_x_{m}{suffix}.npy")).reshape(m, n, 100).to(device)
#         u = torch.tensor(np.load(f"data/ricker_u_{m}{suffix}.npy")).reshape(m, n, 100).to(device)
    
#     timer.start()

#     # x = x.reshape(m, n, 100).to(device) unnecessary line
#     # theta = theta.to(device)
#     # u = u.to(device)
#     density_estimator = inference.append_simulations(theta, x.unsqueeze(1), u).train(
#         distance=distance, 
#         x_obs=obs_cont, 
#         beta=beta,
#         sample_size=sample_size,
#         training_batch_size=64)

#     timer.lap()

#     # increase the prior range in case we can't generate thetas for mis-specified observation
#     prior_new = [Uniform(2 * torch.ones(1), 8 * torch.ones(1)),
#                  Uniform(torch.zeros(1), 80 * torch.ones(1))]
#     prior_new, _, _ = process_prior(prior_new)
#     posterior = inference.build_posterior(density_estimator, prior=prior_new)
    
#     timer.lap()

#     with open(root_name + "/posterior.pkl", "wb") as handle:
#         pickle.dump(posterior, handle)

#     torch.save(sum_net, root_name + "/sum_net.pkl")
#     torch.save(density_estimator, root_name + "/density_estimator.pkl")

#     with open(root_name + "/inference.pkl", "wb") as handle:
#         pickle.dump(inference, handle)
#     timer.stop()


# Methods

For our methods, we compare the proposed robust summary network with mmd-efficient and qmc.

## Experimental parameters
### Helpers for loading data

In [117]:
def load_x_and_theta(simulator, num_simulations, num_realizations, t, load_u=False, qmc=False, _device=device, data_root="../data/"):
    x = torch.tensor(np.load(os.path.join(data_root, f"{simulator}_x_{num_simulations}{'_qmc' if qmc else ''}.npy"))) \
            .reshape(num_simulations, num_realizations, t) \
            .to(device=_device)
    theta = np.load(os.path.join(data_root, f"{simulator}_theta_{num_simulations}{'_qmc' if qmc else ''}.npy"))

    u_path = os.path.join(data_root, f"{simulator}_u_{num_simulations}{'_qmc' if qmc else ''}.npy")
    if (load_u):
        u = torch.tensor(np.load(u_path)).to(device=_device)
        return x, theta, u
    else:
        return x, theta

### Parameters

In [118]:
sample_sizes = [512, 256, 128, 64, 32]
M = 1024 # number of simulations
N = 128 # number of realizations
# batch_size = 256 --- replaced by sample sizes
t_ricker = 100
simulators = ["ricker", "oup"]
samplers = ["qmc", "mc"]

## Experiment 1: Sample-efficient MMD on Ricker ABC

For SBI, the choice of summary statistics is crucial to specify the proper distribution for data. For instance, a Gaussian distribution is poorly specified to bimodal data, but with only mean and variance, there may be a distribution of each type where those summary statistics match. Thus, robust SBI trains a model summarizer to minimize the MMD. 

Computing MMD as a loss function is computationally expensive. We aim to optimize this by using sample-efficient MMD to improve the overall runtime. 

For this experiment, we use approximate Bayesian computation (ABC) as the SBI method of interest. We evaluate three approaches for ABC: the normal approach, the robust approach, and the efficient approach.

### Define experiment

In [119]:
def get_abc_posterior(*, x, theta, beta, obs_cont, m, N, u, mmd_metric, sample_size, max_epochs=30, root=".", verbose=False):
    summary_net = solve_abc(x, beta=beta, obs_cont=obs_cont,
                            m=m, N=N, u=u, sample_size=sample_size, t=t_ricker,
                            mmd_metric=mmd_metric, max_epochs=max_epochs,
                            root=root, verbose=verbose)
    
    get_summary = lambda xx : torch.mean(summary_net.forward_encoder(xx).cpu(), dim=1).detach().numpy()

    summary = get_summary(x)
    summary_obs = get_summary(obs_cont)

    posterior = regression_ABC(summary_obs, theta, summary, 0.05)

    return posterior

### Test solver with training loop

#### Experimental parameters

In [122]:
degrees = [0, 0.1, 0.2]
# beta_list = [1, 2, 3, 4, 5, 10] # degrees of regularization
output_root = '../objects/ABC/ricker_consolidated/'
ricker_observations = lambda degree : torch.tensor(np.load(f"../data/ricker_obs_{int(degree * 10)}.npy")).to(device)
n_sims = 10

optimal_beta = 2 # temporary

simulator = simulators[0] # ricker

x, theta = load_x_and_theta(simulator, M, N, t_ricker)
x_qmc, theta_qmc, u_qmc = load_x_and_theta(simulator, M, N, t_ricker, load_u=True, qmc=True)

In [123]:
for ss in sample_sizes:
    for degree in degrees:
        contaminated_observations = ricker_observations(degree)

        for i in range(n_sims):
            print(f"Simulation {i}")
            exp_name = f"degree={degree}/lambda={optimal_beta}/sample_size={ss}/{str(i)}"
            root_name = os.path.join(output_root, exp_name)

            posterior_normal = get_abc_posterior(x=x, theta=theta, beta=0, obs_cont=contaminated_observations,
                                                m=M, N=N, u=None,
                                                mmd_metric=None, sample_size=ss, root=root_name,
                                                verbose=False)
            
            posterior_robust = get_abc_posterior(x=x, theta=theta, beta=optimal_beta, obs_cont=contaminated_observations,
                                                m=M, N=N, u=None,
                                                mmd_metric=metrics.MMD_unweighted, sample_size=ss, root=root_name,
                                                verbose=False)
            
            posterior_efficient = get_abc_posterior(x=x_qmc, theta=theta_qmc, beta=optimal_beta, obs_cont=contaminated_observations,
                                                    m=M, N=N, u=None,
                                                    mmd_metric=metrics.MMD_weighted, sample_size=ss, root=root_name,
                                                    verbose=False)

            if not os.path.exists(root_name):
                os.makedirs(root_name)
            np.save(root_name + '/posterior_normal.npy', posterior_normal)
            np.save(root_name + '/posterior_robust.npy', posterior_robust)
            np.save(root_name + '/posterior_efficient.npy', posterior_efficient)

Simulation 0
Starting timer for normal_lambda=0
--- Timing: normal_lambda=0 ---
Elapsed time: 2.353s
Timer ran for 30 laps
Average time per epoch: 0.078s

Starting timer for MMD_unweighted_lambda=2
--- Timing: MMD_unweighted_lambda=2 ---
Elapsed time: 2.120s
Timer ran for 20 laps
Average time per epoch: 0.106s

Starting timer for MMD_weighted_lambda=2
--- Timing: MMD_weighted_lambda=2 ---
Elapsed time: 3.337s
Timer ran for 28 laps
Average time per epoch: 0.119s

Simulation 1
Starting timer for normal_lambda=0
--- Timing: normal_lambda=0 ---
Elapsed time: 2.054s
Timer ran for 30 laps
Average time per epoch: 0.068s

Starting timer for MMD_unweighted_lambda=2
--- Timing: MMD_unweighted_lambda=2 ---
Elapsed time: 1.621s
Timer ran for 15 laps
Average time per epoch: 0.108s

Starting timer for MMD_weighted_lambda=2
--- Timing: MMD_weighted_lambda=2 ---
Elapsed time: 2.150s
Timer ran for 18 laps
Average time per epoch: 0.119s

Simulation 2
Starting timer for normal_lambda=0
--- Timing: normal

In [ ]:
# FROM ricker_abc: commented out for new experiments

# for degree in degrees:
#     contaminated_observations = ricker_observations(degree)

#     for beta in beta_list:
#         for i in range(n_sims):
#             print(f"Simulation {i}")
#             root_name = os.path.join(output_root, f'degree={degree}/lambda={beta}/' + str(i))

#             ss = 256

#             posterior_normal = get_abc_posterior(x=x, beta=0, obs_cont=contaminated_observations,
#                                                  m=M, N=N, u=None,
#                                                  mmd_metric=None, sample_size=ss, root=root_name,
#                                                  verbose=False)
            
#             posterior_robust = get_abc_posterior(x=x, beta=beta, obs_cont=contaminated_observations,
#                                                  m=M, N=N, u=None,
#                                                  mmd_metric=metrics.MMD_unweighted, sample_size=ss, root=root_name,
#                                                  verbose=False)
            
#             posterior_efficient = get_abc_posterior(x=x_qmc, beta=beta, obs_cont=contaminated_observations,
#                                                     m=M, N=N, u=None,
#                                                     mmd_metric=metrics.MMD_weighted, sample_size=ss, root=root_name,
#                                                     verbose=False)

#             if not os.path.exists(root_name):
#                 os.makedirs(root_name)
#             np.save(root_name + '/posterior_normal.npy', posterior_normal)
#             np.save(root_name + '/posterior_robust.npy', posterior_robust)
#             np.save(root_name + '/posterior_efficient.npy', posterior_efficient)
#             break

## Experiment 2: NPE experiments

run from command line

for no misspec
```
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=512 > beta_2_misspec_0_512_mmd_log.txt
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=512 > beta_2_misspec_0_512_mmd-efficient_log.txt
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=256 > beta_2_misspec_0_256_mmd_log.txt
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=256 > beta_2_misspec_0_256_mmd-efficient_log.txt
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=128 > beta_2_misspec_0_128_mmd_log.txt
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=128 > beta_2_misspec_0_128_mmd-efficient_log.txt
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=64 > beta_2_misspec_0_64_mmd_log.txt
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=64 > beta_2_misspec_0_64_mmd-efficient_log.txt
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=32 > beta_2_misspec_0_32_mmd_log.txt
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.0 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=32 > beta_2_misspec_0_32_mmd-efficient_log.txt
```

for misspec
```
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=512
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=512
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=256
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=256
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=128
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=128
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=64
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=64
python .\exp_ricker.py --distance="mmd" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sample-size=32
python .\exp_ricker.py --distance="mmd-efficient" --beta=2.0 --degree=0.2 --pre-generated-sim --pre-generated-obs --keep-inference --sampling="qmc" --sample-size=32
```